# SHACL 1.2 Node Expressions — User Guide

SHACL 1.2 Node Expressions is an expression language for computing nodes inside a shapes graph. 

Node expressions are used in SHACL in several places:
 - `sh:targetNode` (which nodes a shape applies to), 
 - `sh:expression` (a boolean conformance check), 
 - `sh:values` (a computed property value), 
 - `sh:deactivated` (whether a shape is active at all), and 
 - `sh:rule`'s `sh:subject`/`sh:predicate`/`sh:object` (deriving new triples).

This guide builds up from simple node expression to the full vocabulary, one new concept at a time. 

## How to run this notebook

1. `pip install "git+https://github.com/hidden-graph/starlayer.git"` (or install the three packages editable from a local checkout — see the main user guide).
2. Run cells from top to bottom — later sections reuse the running "Person" example data from earlier ones.

In [1]:
from starlayergraph import StarLayerGraph, Namespace
from starshacl import StarShaclValidator

EX = Namespace("http://example.org/")

## 1. Selecting target nodes

A node expression can be used as the object of a shape's sh:targetNode, to select the list of targets for a a shape.  

In this example we also introduce `shnex:instancesOf` which operates within a node expression similar to sh:targetClass.  

**Example 1**

In [2]:
#all ex:Person must have at least one ex:email. 
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
    ex:bob a ex:Person .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .

    ex:PersonShape a sh:NodeShape ;
      sh:targetNode [ shnex:instancesOf ex:Person ] ;
      sh:property [ sh:path ex:email ; sh:minCount 1 ] .
""", format="turtle12")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("conforms:", result.conforms, "- expect False (alice and bob both targeted, neither has ex:email)")
print("violations:", result.report_text.count("Focus Node:"))

conforms: False - expect False (alice and bob both targeted, neither has ex:email)
violations: 2


### 1.1 Narrowing the target set

Using `shnex:instancesOf` alone does not provide additional flexibility beyone using is a plain `sh:targetClass`.   

Added flexibility shows up once you *combine* expressions. `shnex:filterShape` narrows a candidate set down to only the nodes that conform to a given shape, computing a target set that a fixed `sh:target*` predicate could not express directly. `shnex:nodes` is used in conjunction with `shnex:filterShape` to select the nodes to be filtered.

**Example 1.1**

In [3]:
#all ex:Person 18 and over must have at least one ex:email. 
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:age 25 .
    ex:bob a ex:Person ; ex:age 12 .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .

    ex:AdultShape a sh:NodeShape ;
      sh:targetNode [ shnex:nodes [ shnex:instancesOf ex:Person ] ;
                      shnex:filterShape [ sh:property [ sh:path ex:age ; sh:minInclusive 18 ] ]
                    ] ;
      sh:property [ sh:path ex:email ; sh:minCount 1 ] .
""", format="turtle12")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("conforms:", result.conforms, "- expect False (only adult alice is targeted; bob, a minor, is never checked)")
print("violations:", result.report_text.count("Focus Node:"))

conforms: False - expect False (only adult alice is targeted; bob, a minor, is never checked)
violations: 1


## 2. Constraint checks: using `sh:expression`

`sh:expression` requires a node expression to evaluate to exactly `(true)` for the shape to conform — a way to write a constraint using an arbitrary boolean condition with node expressions. 

For this example, the object of `sh:expression` is set to `true`.  The literal `true` is a literal node expression.  In later examples we will use other node expression functions to compute the value of the `sh:expression`.

Note that sh:expression true and sh:expression (true) are equivalent. 

**Example 2**

In [4]:
#this example will always validate to true when sh:expression (true)
#changing to sh:expression (false) will validate to false. (anythign other that true or (true))
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:age 25 ; ex:friend ex:bob, ex:carol .
    ex:bob a ex:Person ; ex:age 12 .
    ex:carol a ex:Person ; ex:age 30 .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:expression (true) .
""", format="turtle12")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("evaluates to True:", result.conforms, "- expect True")

evaluates to True: True - expect True


### 2.1 Using `shnex:pathValues` with `sh:expression`

`shnex:pathValues` is a node expression that returns the set of values from following the given property path from the focus node. 

There are variety of uses of `shnex:pathValues` that we will explore in examples to follow.  In this example we use `shnex:pathValues` with `sh:expression` to validate a shape.

**Example 2.1**

In [5]:
#this example will always validate to true when ex:carol ex:valid true.
#changing to ex:valid false will validate to false.
# changing to  ex:carol ex:valid true, false will validate false.  (returns a set of true and false. )
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:carol ex:valid true.
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:carol ;
      sh:expression [shnex:pathValues ex:valid ] .
""", format="turtle12")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("evaluates to True:", result.conforms, "- expect True")

evaluates to True: True - expect True


### 2.2 Using special variables: `"value"` and `"focusNode"` with `sh:expression`

When used with `sh:expression`, `shnex:var "focusNode"` reads the shape's focus node, and `shnex:var "value"` reads the *current value node being checked*.  When used within a  `PropertyShape` this allows the checking of values reached via `sh:path`with the focus node.  

The next example puts a `sh:expression` on a property shape for `ex:friend`, so it runs once per friend — each evaluation sees that friend as `"value"` and `ex:alice` as `"focusNode"`, letting the check compare the two: every friend must be younger than alice herself.

We also introduce the use of sparql function `sparql:less-than`.  This node expression takes two node expressions as arguments, each one should return a list of a single value, performs a comparision and returns a boolean.

**Example 2.2**

In [6]:
#the property shape checks to see that all of Alice's friends are younger than Alice.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:age 25 ; ex:friend ex:bob, ex:carol .
    ex:bob a ex:Person ; ex:age 12 .
    ex:carol a ex:Person ; ex:age 30 .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:S a sh:PropertyShape ; sh:targetNode ex:alice ; sh:path ex:friend ;
      sh:expression [ sparql:less-than (
          [ shnex:pathValues ex:age ; shnex:focusNode [ shnex:var "value" ] ]
          [ shnex:pathValues ex:age ; shnex:focusNode [ shnex:var "focusNode" ] ]
      ) ] .
""", format="turtle12")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("conforms:", result.conforms, "- expect False (carol, 30, is older than alice, 25)")
print("value node in the violation report:", [
    result.report_graph.qname(o) for _p, o in
    [(p, o) for _s, p, o in result.report_graph if str(p).endswith("#value")]
])

conforms: False - expect False (carol, 30, is older than alice, 25)
value node in the violation report: ['ex:carol']


## 3. Computed properties: `sh:values`

`sh:values` computes a property shape's value set from a node expression rather than reading it from the graph directly.  When `sh:values` is used with `sh:path` it creates a  *virtual*, on-demand calculated property. 

In this example, `sh:values` is used to calculate the friendCount which is then used in the constraint.  Note that friendCount only exists virtually durign validation and is not added to the data graph.  

We also introduce the node expression `shnex:count` which counts the values returned by the `shnex:pathValues` node expression.

**Example 3**

In [7]:
#Alice must have at least two friends.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:friend ex:bob, ex:carol .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:property [ sh:path ex:friendCount ; sh:datatype xsd:integer ;
                    sh:values [ shnex:count [ shnex:pathValues ex:friend ] ] ;
                    sh:minInclusive 2 ] .
""", format="turtle12")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("computed ex:friendCount >= 2:", result.conforms, "- expect True")
print("ex:friendCount triples actually in the data graph:", len(list(data.triples((None, EX.friendCount, None)))))

computed ex:friendCount >= 2: True - expect True
ex:friendCount triples actually in the data graph: 0


### 3.1 Using `sh:values` with `StarShaclValidator.evaluate()`

In addition to its use in conformance checks as described above `sh:values` can be "computed on demand"  whenever an instance is displayed or queried,.   

StarShacl has included an `evaluate()` method which materializes `sh:values` in a copy of the data graph.  This copy is then disposed when no longer needed.  

**Example 3.1**

In [8]:
#the display only graph generated from `.evaluate` contains the friendCount
#while the underlying data graph remains unchanged.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:friend ex:bob, ex:carol .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:property [ sh:path ex:friendCount ; 
                    sh:values [ shnex:count [ shnex:pathValues ex:friend ] ]  ] .
""", format="turtle12")
display_result = StarShaclValidator().evaluate(data_graph=data, shacl_graph=shapes)
print("friend count:", [v.toPython() for v in display_result.data_graph.objects(EX.alice, EX.friendCount)])
print("original data_graph still has zero ex:friendCount triples:",
      len(list(data.triples((None, EX.friendCount, None)))) == 0)

friend count: [2]
original data_graph still has zero ex:friendCount triples: True


### 3.2 When a stored value already exists for a property created with `sh:values`

Typically a property will be created with `sh:values` when that value is not expected to exist in the graph. If the property already exists, `.evaluate` will create additional value(s).  Often this is not the desired behavior. 


**Example 3.2**

In [9]:
# ex:alice already has a stored ex:friendCount of 99.
#sh:values will create a secon ex:friendCount in the display graph.
conflicting_data = StarLayerGraph()
conflicting_data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:friend ex:bob, ex:carol ; ex:friendCount 99 .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:property [ sh:path ex:friendCount ; 
                    sh:values [ shnex:count [ shnex:pathValues ex:friend ] ]  ] .
""", format="turtle12")

display_result = StarShaclValidator().evaluate(data_graph=conflicting_data, shacl_graph=shapes)
print("ex:friendCount in evaluate()'s output:",
      sorted(v.toPython() for v in display_result.data_graph.objects(EX.alice, EX.friendCount)),
      "- expect [2, 99] (both the stored and the computed value)")
print("the stored 99 is untouched in the original graph:",
      [v.toPython() for v in conflicting_data.objects(EX.alice, EX.friendCount)])

ex:friendCount in evaluate()'s output: [2, 99] - expect [2, 99] (both the stored and the computed value)
the stored 99 is untouched in the original graph: [99]


### 3.3 Using `sh:defaultValue`

`sh:defaultValue` can be used to provide a value to a `sh:path` when it is missing from the graph.

Note that because the default value is only computed for display purposes and never added to the data graph, if a nickname is later added to the data graph, the default value will have no impact.

**Example 3.3**

In [10]:
#if a person has no nickname, fall back to their first name instead.
default_data = StarLayerGraph()
default_data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:firstName "Alexandra" .
    ex:bob a ex:Person ; ex:firstName "Robert" ; ex:nickname "Bobby" .
""", format="turtle12")

default_shapes = StarLayerGraph()
default_shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    ex:S a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:property [ sh:path ex:nickname ; sh:defaultValue [ shnex:pathValues ex:firstName ] ] .
""", format="turtle12")

display_result = StarShaclValidator().evaluate(data_graph=default_data, shacl_graph=default_shapes)
print("alice's nickname:", [v.toPython() for v in display_result.data_graph.objects(EX.alice, EX.nickname)],
      "- expect ['Alexandra'] (no stored nickname, falls back to her first name)")
print("bob's nickname:", [v.toPython() for v in display_result.data_graph.objects(EX.bob, EX.nickname)],
      "- expect ['Bobby'] (his real, stored nickname - the default never gets used)")
print("original data_graph still has zero ex:nickname triples for alice:",
      len(list(default_data.objects(EX.alice, EX.nickname))) == 0)

alice's nickname: ['Alexandra'] - expect ['Alexandra'] (no stored nickname, falls back to her first name)
bob's nickname: ['Bobby'] - expect ['Bobby'] (his real, stored nickname - the default never gets used)
original data_graph still has zero ex:nickname triples for alice: True


## 4. SPARQL built-ins as node expressions

`sparql:` exposes ordinary SPARQL functions and operators as node expressions (77 functions/operators in total).
— string/numeric/date functions, 
- comparison operators, 
- RDF-1.2-aware functions like `sparql:isTriple`/`sparql:subject`/`sparql:predicate`/`sparql:object` (77 functions/operators in total).

**Every `sparql:` function call takes its arguments as a list, in parentheses, even for a single argument** — `sparql:strlen ( "hi" )`, not `sparql:strlen "hi"`.

**Example 4**

In [11]:
#use sparql:strlen and sparql:greater-than to 
#test alice's name length
data = StarLayerGraph()
data.parse(data="""
@prefix ex: <http://example.org/> . 
ex:alice ex:name "Alice" .""", format="turtle12")



shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:expression [ sparql:greater-than (
          [ sparql:strlen ( [ shnex:pathValues ex:name ] ) ]
          3
      ) ] .
""", format="turtle12")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("ex:alice's name is longer than 3 characters:", result.conforms, "- expect True")

ex:alice's name is longer than 3 characters: True - expect True


### 4.1 Arithmetic functions

`sparql:plus`/`subtract`/`multiply`/`divide`, and the two unary forms `sparql:unary-plus`/`unary-minus`, are SPARQL's ordinary arithmetic operators, available as node expressions.

The oprators are used with sh:values to create a property for display purposes when `.evaluate()` is called on the SHACL validator.


**Example 4.1**

In [12]:
#use sh:values to compute each arithmetic result as its own virtual property,
#then use evaluate() before outputting the graph.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:x 9 .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:property [ sh:path ex:v_sum ; sh:values [ sparql:plus ( [ shnex:pathValues ex:x ] 3 ) ] ] ;
      sh:property [ sh:path ex:v_difference ; sh:values [ sparql:subtract ( [ shnex:pathValues ex:x ] 3 ) ] ] ;
      sh:property [ sh:path ex:v_product ; sh:values [ sparql:multiply ( [ shnex:pathValues ex:x ] 3 ) ] ] ;
      sh:property [ sh:path ex:v_quotient ; sh:values [ sparql:divide ( [ shnex:pathValues ex:x ] 3 ) ] ] ;
      sh:property [ sh:path ex:v_negatedThree ; sh:values [ sparql:unary-minus ( 3 ) ] ] ;
      sh:property [ sh:path ex:v_positiveX ; sh:values [ sparql:unary-plus ( [ shnex:pathValues ex:x ] ) ] ] .
""", format="turtle12")

display_result = StarShaclValidator().evaluate(data_graph=data, shacl_graph=shapes)
print(display_result.data_graph.serialize(format="turtle12"))

@prefix ex: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:alice ex:v_difference 6 ;
    ex:v_negatedThree -3 ;
    ex:v_positiveX 9 ;
    ex:v_product 27 ;
    ex:v_quotient 3.0 ;
    ex:v_sum 12 ;
    ex:x 9 ;
    a ex:Person .



### 4.2 Numeric, date/time, hash & term-type functions

- `sparql:abs` — absolute value of a number.
- `sparql:round` — rounds a number to the nearest integer.
- `sparql:year` — extracts the year component from a date/dateTime.
- `sparql:md5` — MD5 hash of a string, returned as a hex string.
- `sparql:isNumeric` — true if the value is a numeric literal.
- `sparql:bnode`/`isBlank` — `bnode` constructs a fresh blank node; `isBlank` checks whether a value is one.
- `sparql:strdt` — builds a literal from a lexical string and an explicit datatype IRI.
- `sparql:datatype` — the datatype IRI of a literal.
- `sparql:lang` — the language tag of a language-tagged literal (e.g. `"en"` from `"hi"@en`).

As above in 4.1 the operators are used with sh:values to create a property for display purposes when `.evaluate()` is called on the SHACL validator.

**Example 4.2**

In [13]:
#use sh:values to compute each numeric/date/hash/term-type result as its own
#virtual property.
#then use evaluate() before outputting the graph.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:alice a ex:Person ;
      ex:score -42 ;
      ex:price 3.7 ;
      ex:createdAt "2023-12-25T10:30:00"^^xsd:dateTime ;
      ex:password "hello" ;
      ex:idString "42" ;
      ex:greeting "hi"@en .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:property [ sh:path ex:v_absScore ; sh:values [ sparql:abs ( [ shnex:pathValues ex:score ] ) ] ] ;
      sh:property [ sh:path ex:v_roundedPrice ; sh:values [ sparql:round ( [ shnex:pathValues ex:price ] ) ] ] ;
      sh:property [ sh:path ex:v_createdYear ; sh:values [ sparql:year ( [ shnex:pathValues ex:createdAt ] ) ] ] ;
      sh:property [ sh:path ex:v_passwordHash ; sh:values [ sparql:md5 ( [ shnex:pathValues ex:password ] ) ] ] ;
      sh:property [ sh:path ex:v_scoreIsNumeric ; sh:values [ sparql:isNumeric ( [ shnex:pathValues ex:score ] ) ] ] ;
      sh:property [ sh:path ex:v_freshBnodeIsBlank ; sh:values [ sparql:isBlank ( [ sparql:bnode () ] ) ] ] ;
      sh:property [ sh:path ex:v_idAsInteger ; sh:values [ sparql:strdt ( [ shnex:pathValues ex:idString ] xsd:integer ) ] ] ;
      sh:property [ sh:path ex:v_scoreDatatype ; sh:values [ sparql:datatype ( [ shnex:pathValues ex:score ] ) ] ] ;
      sh:property [ sh:path ex:v_greetingLang ; sh:values [ sparql:lang ( [ shnex:pathValues ex:greeting ] ) ] ] .
""", format="turtle12")

display_result = StarShaclValidator().evaluate(data_graph=data, shacl_graph=shapes)
print(display_result.data_graph.serialize(format="turtle12"))

@prefix ex: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:alice ex:createdAt "2023-12-25T10:30:00"^^xsd:dateTime ;
    ex:greeting "hi"@en ;
    ex:idString "42" ;
    ex:password "hello" ;
    ex:price 3.7 ;
    ex:score -42 ;
    ex:v_absScore 42 ;
    ex:v_createdYear 2023 ;
    ex:v_freshBnodeIsBlank true ;
    ex:v_greetingLang "en" ;
    ex:v_idAsInteger 42 ;
    ex:v_passwordHash "5d41402abc4b2a76b9719d911017c592" ;
    ex:v_roundedPrice 4.0 ;
    ex:v_scoreDatatype xsd:integer ;
    ex:v_scoreIsNumeric true ;
    a ex:Person .



### 4.3 RDF-1.2 triple-term functions
This is the part of the `sparql:` vocabulary most distinctive to an RDF 1.2 implementation where triple terms can be the object of a triple.

- `sparql:isTriple` — true if the value is an RDF-1.2 triple term.
- `sparql:triple` — constructs a triple term from a subject, predicate, and object.
- `sparql:subject` — the subject of a triple term.
- `sparql:predicate` — the predicate of a triple term.
- `sparql:object` — the object of a triple term.

As above the operators are used with `sh:values` to create a property for display purposes when `.evaluate()` is called on the SHACL validator.

**Example 4.3**

In [14]:
#use sh:values to compute each triple-term result as its own virtual
#property, then use evaluate() before outputting the graph.

data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:claim1 rdf:reifies <<( ex:bob ex:knows ex:carol )>> .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:claim1 ;
      sh:property [ sh:path ex:v_claimIsTriple ; sh:values [ sparql:isTriple ( [ shnex:pathValues rdf:reifies ] ) ] ] ;
      sh:property [ sh:path ex:v_claimSubject ; sh:values [ sparql:subject ( [ shnex:pathValues rdf:reifies ] ) ] ] ;
      sh:property [ sh:path ex:v_claimPredicate ; sh:values [ sparql:predicate ( [ shnex:pathValues rdf:reifies ] ) ] ] ;
      sh:property [ sh:path ex:v_claimObject ; sh:values [ sparql:object ( [ shnex:pathValues rdf:reifies ] ) ] ] ;
      sh:property [ sh:path ex:v_constructedTriple ; sh:values [ sparql:triple ( ex:s ex:p ex:o ) ] ] ;
      sh:property [ sh:path ex:v_constructedSubject ; sh:values [ sparql:subject ( [ sparql:triple ( ex:s ex:p ex:o ) ] ) ] ] .
""", format="turtle12")

display_result = StarShaclValidator().evaluate(data_graph=data, shacl_graph=shapes)

print(display_result.data_graph.serialize(format="turtle12"))

@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:claim1 ex:v_claimIsTriple true ;
    ex:v_claimObject ex:carol ;
    ex:v_claimPredicate ex:knows ;
    ex:v_claimSubject ex:bob ;
    ex:v_constructedSubject ex:s ;
    ex:v_constructedTriple <<( ex:s ex:p ex:o )>> ;
    rdf:reifies <<( ex:bob ex:knows ex:carol )>> .



### 4.4 RDF-1.2 directional language string functions

Additional `sparql:` functions related to RDF 1.2 functionality to handle directed language strings.

- `sparql:hasLangdir` — true if the value is an RDF-1.2 directional language string (`"..."@lang--dir`).
- `sparql:langdir` — the text direction (`"ltr"`/`"rtl"`) of a directional language string.
- `sparql:strlangdir` — builds a directional language string from a plain string, a language tag, and a direction.

As above, the operators are used with `sh:values` to create a property for display purposes when `.evaluate()` is called on the SHACL validator.

**Example 4.4**

In [15]:
#use sh:values to compute each directional-language-string result as its
#own virtual property, then use evaluate() before outputting the graph.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:greeting "hi"@en--ltr .
    ex:bob a ex:Person ; ex:greeting "hi"@en .
    ex:carol a ex:Person .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:S a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:property [ sh:path ex:v_greetingHasDir ;
                    sh:values [ sparql:hasLangdir ( [ shnex:pathValues ex:greeting ] ) ] ] ;
      sh:property [ sh:path ex:v_greetingDir ;
                    sh:values [ sparql:langdir ( [ shnex:pathValues ex:greeting ] ) ] ] ;
      sh:property [ sh:path ex:v_constructedGreeting ;
                    sh:values [ sparql:strlangdir ( "hello" "en" "ltr" ) ] ] .
""", format="turtle12")

display_result = StarShaclValidator().evaluate(data_graph=data, shacl_graph=shapes)
print(display_result.data_graph.serialize(format="turtle12"))

@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:alice ex:greeting "hi"@en--ltr ;
    ex:v_constructedGreeting "hello"@en--ltr ;
    ex:v_greetingDir "ltr" ;
    ex:v_greetingHasDir true ;
    a ex:Person .

ex:bob ex:greeting "hi"@en ;
    ex:v_constructedGreeting "hello"@en--ltr ;
    ex:v_greetingDir "" ;
    ex:v_greetingHasDir false ;
    a ex:Person .

ex:carol ex:v_constructedGreeting "hello"@en--ltr ;
    a ex:Person .



### 4.5 Additional sparql: functions

- `sparql:bound` — true if a node expression produced a value at all (not unbound/missing).
- `sparql:coalesce` — returns the first bound value among several alternatives, skipping unbound ones.
- `sparql:if` — a conditional: evaluates a condition, then returns one of two branches.

As above, the operators are used with `sh:values` to create a property for display purposes when `.evaluate()` is called on the SHACL validator.

**Example 4.5**

In [16]:
#use sh:values to compute each special-form result as its own virtual
#property, then use evaluate() before outputting the graph.
#alice has a phone, bob only an email, carol neither - showing how
#sparql:coalesce/bound behave differently per person.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:age 25 ; ex:phone "555-1234" .
    ex:bob a ex:Person ; ex:age 12 ; ex:email "bob@example.org" .
    ex:carol a ex:Person ; ex:age 30 .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:S a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:property [ sh:path ex:v_ageCategory ;
                    sh:values [ sparql:if (
                      [ sparql:greater-than-or-equal ( [ shnex:pathValues ex:age ] 18 ) ] 
                      "adult" 
                      "minor" ) ] ] ;
      sh:property [ sh:path ex:v_contactMethod ;
                    sh:values [ sparql:coalesce ( [ shnex:pathValues ex:phone ] [ shnex:pathValues ex:email ] "none" ) ] ] ;
      sh:property [ sh:path ex:v_hasPhone ;
                    sh:values [ sparql:bound ( [ shnex:pathValues ex:phone ] ) ] ] .
""", format="turtle12")

display_result = StarShaclValidator().evaluate(data_graph=data, shacl_graph=shapes)
print(display_result.data_graph.serialize(format="turtle12"))

@prefix ex: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:alice ex:age 25 ;
    ex:phone "555-1234" ;
    ex:v_ageCategory "adult" ;
    ex:v_contactMethod "555-1234" ;
    ex:v_hasPhone true ;
    a ex:Person .

ex:bob ex:age 12 ;
    ex:email "bob@example.org" ;
    ex:v_ageCategory "minor" ;
    ex:v_contactMethod "bob@example.org" ;
    ex:v_hasPhone false ;
    a ex:Person .

ex:carol ex:age 30 ;
    ex:v_ageCategory "adult" ;
    ex:v_contactMethod "none" ;
    ex:v_hasPhone false ;
    a ex:Person .



### 4.6 Comparison, logical & string functions

- `sparql:not-equals` — true if two values are not equal.
- `sparql:logical-and` — true if both boolean operands are true.
- `sparql:sameValue` — true if two values are value-equal, even across different datatypes (e.g. `1` and `1.0`).
- `sparql:sameTerm` — true only if two values are the exact same RDF term (same lexical form and datatype) - stricter than `sameValue`.
- `sparql:contains` — true if a string contains a given substring.
- `sparql:replace` — replaces occurrences of a pattern in a string.
- `sparql:regex` — true if a string matches a regular expression.
- `sparql:strstarts` — true if a string starts with a given prefix.

As above, the operators are used with `sh:values` to create a property for display purposes when `.evaluate()` is called on the SHACL validator.

**Example 4.6**

In [17]:
#use sh:values to compute each comparison/logical/string result as its
#own virtual property, then use evaluate() before outputting the graph.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:x 10 ; ex:y 3 ; ex:name "Alice Munro" .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:property [ sh:path ex:v_notEqual ; sh:values [ sparql:not-equals ( [ shnex:pathValues ex:x ] [ shnex:pathValues ex:y ] ) ] ] ;
      sh:property [ sh:path ex:v_bothInRange ; sh:values [ sparql:logical-and ( [ sparql:greater-than ( [ shnex:pathValues ex:x ] 5 ) ] [ sparql:less-than ( [ shnex:pathValues ex:y ] 5 ) ] ) ] ] ;
      sh:property [ sh:path ex:v_sameValueCheck ; sh:values [ sparql:sameValue ( 1 1.0 ) ] ] ;
      sh:property [ sh:path ex:v_sameTermCheck ; sh:values [ sparql:sameTerm ( 1 1.0 ) ] ] ;
      sh:property [ sh:path ex:v_containsMunro ; sh:values [ sparql:contains ( [ shnex:pathValues ex:name ] "Munro" ) ] ] ;
      sh:property [ sh:path ex:v_replacedName ; sh:values [ sparql:replace ( [ shnex:pathValues ex:name ] "Munro" "Smith" ) ] ] ;
      sh:property [ sh:path ex:v_matchesAlicePattern ; sh:values [ sparql:regex ( [ shnex:pathValues ex:name ] "^Alice" ) ] ] ;
      sh:property [ sh:path ex:v_startsWithAlice ; sh:values [ sparql:strstarts ( [ shnex:pathValues ex:name ] "Alice" ) ] ] .
""", format="turtle12")

display_result = StarShaclValidator().evaluate(data_graph=data, shacl_graph=shapes)
print(display_result.data_graph.serialize(format="turtle12"))

@prefix ex: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:alice ex:name "Alice Munro" ;
    ex:v_bothInRange true ;
    ex:v_containsMunro true ;
    ex:v_matchesAlicePattern true ;
    ex:v_notEqual true ;
    ex:v_replacedName "Alice Smith" ;
    ex:v_sameTermCheck false ;
    ex:v_sameValueCheck true ;
    ex:v_startsWithAlice true ;
    ex:x 10 ;
    ex:y 3 .


## 5. The `shnex:` vocabulary

`shnex:` is SHACL 1.2 Node Expressions' own native vocabulary — operators the specification defines directly, for selecting, reading, aggregating, and transforming nodes.

`shnex:`  operators are distinct from `sparql:` (section 4), which expose ordinary SPARQL functions and operators as node expressions. Between the two namespaces, `shnex:` and `sparql:` make up the full node-expression language.

`shnex:` operators have already appeared in this guide as they came up naturally: `shnex:instancesOf`/`filterShape` for selecting target nodes (section 1), `shnex:pathValues`/`var` for reading values via a property path and referencing special variables (section 2), and `shnex:count` for aggregating a property's values (section 3). 

This section works through the full `shnex:` vocabulary as a complete refence.  

### 5.1 Control flow & list transformations

`shnex:if`/`then`/`else`; `shnex:distinct`/`remove`/`intersection`/`concat` (set/list combination, using RDF 1.2's own exact *term*-equality, not value-equality — a genuinely different literal, even one that's numerically equal, is never treated as a duplicate); `shnex:orderBy`/`desc` and `shnex:limit`/`offset` (sorting and paging a node list); `shnex:flatMap` (evaluate an expression once per node, concatenating the results).

The operators are used with `sh:values` to create a property for display purposes when `.evaluate()` is called on the SHACL validator.

**Example 5.1**

In [18]:
#use sh:values to compute each control-flow/list-transformation result as
#its own virtual property, then use evaluate() before outputting the graph.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:friend ex:bob, ex:carol .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:property [ sh:path ex:v_ifResult ;
                    sh:values [ shnex:if [ sparql:equals ( [ shnex:count [ shnex:pathValues ex:friend ] ] 2 ) ] ;
                                 shnex:then true ; shnex:else false ] ] ;
      sh:property [ sh:path ex:v_distinctCount ;
                    sh:values [ shnex:count [ shnex:distinct ( 4 2 4 ) ] ] ] ;
      sh:property [ sh:path ex:v_minAfterRemove ;
                    sh:values [ shnex:min [ shnex:remove ( 3 2 ) ; shnex:nodes ( 1 2 3 4 ) ] ] ] ;
      sh:property [ sh:path ex:v_maxAfterRemove ;
                    sh:values [ shnex:max [ shnex:remove ( 3 2 ) ; shnex:nodes ( 1 2 3 4 ) ] ] ] ;
      sh:property [ sh:path ex:v_intersectionSum ;
                    sh:values [ shnex:sum [ shnex:intersection ( ( 4 3 2 1 ) ( 3 2 ) ) ] ] ] ;
      sh:property [ sh:path ex:v_concatSum ;
                    sh:values [ shnex:sum [ shnex:concat ( ( 5 4 3 ) ( 2 1 ) ) ] ] ] ;
      sh:property [ sh:path ex:v_orderByDescLimit ;
                    sh:values [ shnex:limit 1 ;
                                shnex:nodes [ shnex:orderBy [ shnex:var "focusNode" ] ;
                                              shnex:nodes ( 8 2 3 ) ; shnex:desc true ] ] ] ;
      sh:property [ sh:path ex:v_maxWithLimit ;
                    sh:values [ shnex:max [ shnex:nodes ( 1 2 3 4 ) ; shnex:limit 2 ] ] ] ;
      sh:property [ sh:path ex:v_minWithOffset ;
                    sh:values [ shnex:min [ shnex:nodes ( 1 2 3 4 ) ; shnex:offset 2 ] ] ] ;
      sh:property [ sh:path ex:v_flatMapSum ;
                    sh:values [ shnex:sum [ shnex:nodes ( 1 2 3 ) ; shnex:flatMap [ shnex:var "focusNode" ] ] ] ] .
""", format="turtle12")

display_result = StarShaclValidator().evaluate(data_graph=data, shacl_graph=shapes)
print(display_result.data_graph.serialize(format="turtle12"))

@prefix ex: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:alice ex:friend ex:carol, ex:bob ;
    ex:v_concatSum 15 ;
    ex:v_distinctCount 2 ;
    ex:v_flatMapSum 6 ;
    ex:v_ifResult true ;
    ex:v_intersectionSum 5 ;
    ex:v_maxAfterRemove 4 ;
    ex:v_maxWithLimit 2 ;
    ex:v_minAfterRemove 1 ;
    ex:v_minWithOffset 3 ;
    ex:v_orderByDescLimit 8 ;
    a ex:Person .


### 5.2 Extra aggregates & shape-matching helpers

`shnex:min`/`max` (alongside `count`, section 3, and `sum`, section 7); `shnex:findFirst` (the first node in a list conforming to a given shape); `shnex:matchAll` (do *all* nodes conform); `shnex:conformsToShape` (does one specific node conform); `shnex:nodesMatching` (every node in the whole data graph conforming to a shape).

The operators are used with `sh:values` to create a property for display purposes when `.evaluate()` is called on the SHACL validator.

**Example 5.2**

In [19]:
#use sh:values to compute each aggregate/shape-matching result as its own
#virtual property, then use evaluate() before outputting the graph.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
    ex:bob a ex:Person .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:AtLeastThreeShape a sh:NodeShape ; sh:minInclusive 3 .
    ex:PersonShape a sh:NodeShape ; sh:hasValue ex:bob .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:property [ sh:path ex:v_min ; sh:values [ shnex:min ( 4 5 3 ) ] ] ;
      sh:property [ sh:path ex:v_max ; sh:values [ shnex:max ( 4 5 3 ) ] ] ;
      sh:property [ sh:path ex:v_findFirst ;
                    sh:values [ shnex:findFirst ex:AtLeastThreeShape ; shnex:nodes ( 2 1 4 3 5 ) ] ] ;
      sh:property [ sh:path ex:v_matchAll ;
                    sh:values [ shnex:matchAll ex:AtLeastThreeShape ; shnex:nodes ( 4 3 5 ) ] ] ;
      sh:property [ sh:path ex:v_conformsToShape ;
                    sh:values [ shnex:conformsToShape ( ex:bob ex:PersonShape ) ] ] ;
      sh:property [ sh:path ex:v_nodesMatchingCount ;
                    sh:values [ shnex:count [ shnex:nodesMatching ex:PersonShape ] ] ] .
""", format="turtle12")

display_result = StarShaclValidator().evaluate(data_graph=data, shacl_graph=shapes)
print(display_result.data_graph.serialize(format="turtle12"))

@prefix ex: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:alice ex:v_conformsToShape true ;
    ex:v_findFirst 4 ;
    ex:v_matchAll true ;
    ex:v_max 5 ;
    ex:v_min 3 ;
    ex:v_nodesMatchingCount 1 ;
    a ex:Person .

ex:bob a ex:Person .



## 6. Dynamic activation: with `sh:deactivated`

`sh:deactivated` can also hold a node expression. It is evaluated once **per focus node** so it acts as a dynamic *per-target* filter, not just a shape-wide on/off switch: a shape can be active for some targets and deactivated for others in the same validator call.

**Example 6**

In [20]:
#non-legacy accounts are validated against the Adults-only shape
#while non-legacy accounts are deactivated for this shape.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:age 12 ; ex:legacyAccount true .
    ex:dave a ex:Person ; ex:age 12 ; ex:legacyAccount false .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .

    ex:AdultOnlyShape a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:deactivated [ shnex:exists [ shnex:filterShape ex:IsLegacy ;
                                      shnex:nodes [ shnex:var "focusNode" ] ] ] ;
      sh:property [ sh:path ex:age ; sh:minInclusive 18 ] .
    ex:IsLegacy a sh:NodeShape ; sh:property [ sh:path ex:legacyAccount ; sh:hasValue true ] .
""", format="turtle12")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
# ex:alice (a legacy account) is deactivated for herself specifically -
# despite being 12, she's never checked. ex:dave (not legacy) still is.
print("conforms:", result.conforms, "- expect False")
print("violations:", result.report_text.count("Focus Node:"), "(only dave, not alice)")

conforms: False - expect False
violations: 1 (only dave, not alice)


## 7. Extending the vocabulary yourself: custom node expression functions

Beyond the built-in `shnex:`/`sparql:` operators, a shapes graph can declare its own reusable node-expression functions ("Custom Node Expressions"). Two forms exist - each shown below.

### 7.1 Custom List Parameter Functions

A `sh:ListParameterExpressionFunction` is called using the function's IRI, with a list of argument node expressions. Arguments are read inside `sh:bodyExpression` positionally, via `[ shnex:arg 0 ]`, `[ shnex:arg 1 ]`, ...

**Example 7.1**

In [21]:
# Custom List Parameter Function: ex:spacedConcat(a, b) -> "a b"
#used with sh:values + evaluate() - we haven't covered sh:rule yet.
data2 = StarLayerGraph()
data2.parse(data="@prefix ex: <http://example.org/> . ex:alice a ex:Thing .", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:spacedConcat a sh:ListParameterExpressionFunction ;
      sh:bodyExpression [ sparql:concat ( [ shnex:arg 0 ] " " [ shnex:arg 1 ] ) ] ;
      sh:parameter [ a sh:Parameter ; sh:path shnex:arg0 ; sh:name "first string" ] ;
      sh:parameter [ a sh:Parameter ; sh:path shnex:arg1 ; sh:name "second string" ] .

    ex:R a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:property [ sh:path ex:v_greeting ; sh:values [ ex:spacedConcat ( "hello" "world" ) ] ] .
""", format="turtle12")

display_result = StarShaclValidator().evaluate(data_graph=data2, shacl_graph=shapes)
print(display_result.data_graph.serialize(format="turtle12"))

@prefix ex: <http://example.org/> .

ex:alice ex:v_greeting "hello world" ;
    a ex:Thing .



### 7.2 Custom Named Parameter Functions

`sh:NamedParameterExpressionFunction` - is the custom "named node expression" form.  

The function is defined as an instance of sh:NamedParameterExpressionFunction with at least one parameter.  

Parameters are separately defined as an instance of sh:Parameter.  One of the defined parameters must be set to sh:keyParameter true.  It is the use of this key parameter as a node expression that triggers the evaluation by the named parameter function.  

A few points to note:
- the key parameter is identified by the value of its sh:path property.  (ex:payroll_rate and ex:totalPayroll_payrolls in the examples below.). These parameters must be unique across the set of named parameter functions, since they are used to map the use of the function in the shape, to the function where they are used.  If not unique there would exist confusion as to what expression to apply.
- StarShaclValidator().evaluate evaluates the data graph against the shapes graph in a single pass, populating values.  For our example, the individual employee payroll needs to be calculated first (pass1) before calculating total payroll (pass2). This drives the need to perform two evaluations. 



**Example 7.2**

In [22]:
# This example uses two named parameter functions to first calculate
# payroll for individual employees and then total payroll expense
# for the company.
# The graph pass1 is used to hold the evaluated graph after applying
# the ex:PayrollExpression to calculate individual employee payroll.
# Note that ex:PayrollExpression uses two parameters, with ex:payroll_rate
# being the key parameter used to trigger the function.
# pass1 is then used as the input to pass2, which uses ex:TotalPayrollExpression
# to calculate total payroll for the company after applying an overhead rate.

data5 = StarLayerGraph()
data5.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:ACME ex:employee ex:bob, ex:carol ;
            ex:overhead 0.20 .
    ex:bob ex:rate 20 ; ex:hours 10 .
    ex:carol ex:rate 15 ; ex:hours 20 .
""", format="turtle12")

shapes_payroll = StarLayerGraph()
shapes_payroll.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:PayrollExpression a sh:NamedParameterExpressionFunction ;
      sh:parameter ex:PayrollExpression-rate, ex:PayrollExpression-hours ;
      sh:bodyExpression [ sparql:multiply ( [ shnex:arg ex:payroll_rate ] [ shnex:arg ex:payroll_hours ] ) ] .
    ex:PayrollExpression-rate a sh:Parameter ; sh:path ex:payroll_rate ; sh:keyParameter true .
    ex:PayrollExpression-hours a sh:Parameter ; sh:path ex:payroll_hours .

    ex:PerEmployeePay a sh:NodeShape ; sh:targetObjectsOf ex:employee ;
      sh:property [ sh:path ex:v_payroll ;
                    sh:values [ ex:payroll_rate [ shnex:pathValues ex:rate ] ;
                                ex:payroll_hours [ shnex:pathValues ex:hours ] ] ] .
""", format="turtle12")

shapes_total = StarLayerGraph()
shapes_total.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    @prefix sparql: <http://www.w3.org/ns/sparql#> .

    ex:TotalPayrollExpression a sh:NamedParameterExpressionFunction ;
      sh:parameter ex:TotalPayrollExpression-payrolls ;
      sh:bodyExpression [ sparql:multiply (
          [ shnex:sum [ shnex:arg ex:totalPayroll_payrolls ] ]
          [ sparql:plus ( 1 [ shnex:pathValues ex:overhead ] ) ]
      ) ] .

    ex:TotalPayrollExpression-payrolls a sh:Parameter ; sh:path ex:totalPayroll_payrolls ; sh:keyParameter true .

    ex:TotalPayroll a sh:NodeShape ; sh:targetNode ex:ACME ;
      sh:property [ sh:path ex:v_totalPayroll ;
                    sh:values [ ex:totalPayroll_payrolls [ shnex:pathValues ( ex:employee ex:v_payroll ) ] ] ] .
""", format="turtle12")

pass1 = StarShaclValidator().evaluate(data_graph=data5, shacl_graph=shapes_payroll)
pass2 = StarShaclValidator().evaluate(data_graph=pass1.data_graph, shacl_graph=shapes_total)
print(pass2.data_graph.serialize(format="turtle12"))

@prefix ex: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:ACME ex:employee ex:carol, ex:bob ;
    ex:overhead 0.20 ;
    ex:v_totalPayroll 600.00 .

ex:bob ex:hours 10 ;
    ex:rate 20 ;
    ex:v_payroll 200 .

ex:carol ex:hours 20 ;
    ex:rate 15 ;
    ex:v_payroll 300 .



## 8. `sh:nodeByExpression`: choosing a shape to check via a node expression

`sh:nodeByExpression` behaves like `sh:node` (does the focus node also conform to a second, referenced shape?), except the referenced shape itself is the result of evaluating a node expression rather than a fixed IRI. The constant form shown here is the trivial case — an ordinary IRI is itself already a valid node expression — but a more elaborate deployment could compute *which* shape to check per focus node.

**Example 8**

In [23]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:validPerson a ex:Person, ex:Verified .
    ex:invalidPerson a ex:Person .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .

    ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:nodeByExpression ex:VerifiedShape .
    ex:VerifiedShape a sh:NodeShape ; sh:class ex:Verified .
""", format="turtle12")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("conforms:", result.conforms, "- expect False (ex:invalidPerson is a Person but not Verified)")
print("violations:", result.report_text.count("Focus Node:"))

conforms: False - expect False (ex:invalidPerson is a Person but not Verified)
violations: 1


## Further reading

- `packages/shacl/docs/shacl12-gap-matrix.md` — full term-by-term status of every `shnex:`/`sparql:` operator, both custom-function forms, and every SHACL integration point (`sh:targetNode`, `sh:expression`, `sh:values`, `sh:defaultValue`, `sh:deactivated`, `sh:nodeByExpression`) against the live W3C draft, including known limitations.
- `packages/shacl/tests/integration/test_shnex_node_expressions.py`, `test_custom_node_expression_functions.py`, `test_node_expression_integration_points.py`, `test_evaluate_virtual_values.py`, `test_sh_values.py`, `test_node_by_expression.py` — the full test suite this guide's examples are drawn from. In particular, `test_shnex_node_expressions.py`'s `SPARQL_FUNCTION_PIPELINE_CASES` table exercises **every** `sparql:` function/operator (not just the representative sample shown in sections 4 and 5 above) through the real `validate()` pipeline.
- **Three processing modes**: `validate()` (checks conformance, never mutates), `apply_rules()` (executes `sh:rule`, materializes real triples), `evaluate()` (computes `sh:values`-declared virtual properties into a throwaway merged graph, never mutates). pySHACL itself only has a notion of the first two — `evaluate()` is `starshacl`'s own addition, built because `sh:values`'s "computed only on demand" semantics had no way to be read directly otherwise.
- **`sh:values`/`sh:defaultValue` together**: a property shape's effective value set is the *union* of its real, path-based values and its `sh:values`-computed values; `sh:defaultValue` only fills in as a last resort, when that union is still empty — never overriding a real or computed value. See section 3 above and `_patch_shape_value_nodes_for_sh_values` in `starshacl/validator.py` for the exact three-step algorithm, quoted from SHACL 1.2 Core's own "Value Nodes of Property Shapes" section.
- **Not yet supported**: calling a custom node expression function as an ordinary SPARQL function by name from `sh:select`/`sh:sparqlExpr` query text (e.g. `ex:instanceCount(ex:Name)` inside a `BIND(...)`) — a separate, larger mechanism from the node-expression call forms shown above.
- **Deliberately not implemented**: "Dynamic SHACL" — the spec's own optional dialect letting *any* constraint parameter (`sh:minInclusive`, `sh:in`, `sh:class`, ...) be computed via a node expression, framed by the spec itself as something implementations "MAY support", not a requirement.